In [28]:
# !pip install scikit-learn pandas catboost imblearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss
)

df = pd.read_csv('data/mxmh_survey_results.csv')
print(df.columns)

Index(['Timestamp', 'Age', 'Primary streaming service', 'Hours per day',
       'While working', 'Instrumentalist', 'Composer', 'Fav genre',
       'Exploratory', 'Foreign languages', 'BPM', 'Frequency [Classical]',
       'Frequency [Country]', 'Frequency [EDM]', 'Frequency [Folk]',
       'Frequency [Gospel]', 'Frequency [Hip hop]', 'Frequency [Jazz]',
       'Frequency [K pop]', 'Frequency [Latin]', 'Frequency [Lofi]',
       'Frequency [Metal]', 'Frequency [Pop]', 'Frequency [R&B]',
       'Frequency [Rap]', 'Frequency [Rock]', 'Frequency [Video game music]',
       'Anxiety', 'Depression', 'Insomnia', 'OCD', 'Music effects',
       'Permissions'],
      dtype='str')


In [26]:
"""
Reduce a large genre set to k combined "super-genres" and model
P(listen to super-genre | 4 mental-health scores).

Replace the X / genre_matrix loading block with your real data.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.calibration import calibration_curve

# ---------- 1. Load data ----------
# X: (n_samples, 4) mental health scores, 0-10
# genre_matrix: (n_samples, n_genres) binary listen/not-listen
X = np.random.uniform(0, 10, size=(500, 4))          # placeholder
genre_matrix = np.random.binomial(1, 0.3, size=(500, 20))  # placeholder

# ---------- 2. Cluster genres ----------
def cluster_genres(genre_matrix, k):
    corr = np.nan_to_num(np.corrcoef(genre_matrix.T))
    dist = 1 - corr
    np.fill_diagonal(dist, 0)
    Z = linkage(squareform(dist, checks=False), method="average")
    return fcluster(Z, t=k, criterion="maxclust"), Z

def build_combined_genres(genre_matrix, labels, k):
    combined = np.zeros((genre_matrix.shape[0], k))
    for c in range(1, k + 1):
        cols = np.where(labels == c)[0]
        combined[:, c - 1] = (genre_matrix[:, cols].sum(axis=1) > 0).astype(int)
    return combined

# ---------- 3. Cross-validated scoring for a given k ----------
def evaluate_k(X, genre_matrix, k, n_splits=5):
    labels, _ = cluster_genres(genre_matrix, k)
    Y = build_combined_genres(genre_matrix, labels, k)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs, losses = [], []
    for train_idx, test_idx in kf.split(X):
        model = MultiOutputClassifier(LogisticRegression(max_iter=1000))
        model.fit(X[train_idx], Y[train_idx])
        for j, est in enumerate(model.estimators_):
            probs = est.predict_proba(X[test_idx])[:, 1]
            y_true = Y[test_idx, j]
            if len(np.unique(y_true)) > 1:
                aucs.append(roc_auc_score(y_true, probs))
            losses.append(log_loss(y_true, probs, labels=[0, 1]))
    return np.mean(aucs), np.mean(losses)

# ---------- 4. Sweep k ----------
k_range = range(2, 15)
results = [(k, *evaluate_k(X, genre_matrix, k)) for k in k_range]
results = np.array(results)
best_k = int(results[np.argmax(results[:, 1]), 0])
print(f"Best k by AUC: {best_k}")

fig, ax1 = plt.subplots()
ax1.plot(results[:, 0], results[:, 1], "o-", color="tab:blue", label="AUC")
ax1.set_xlabel("k (number of combined genres)")
ax1.set_ylabel("AUC", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(results[:, 0], results[:, 2], "s-", color="tab:red", label="Log loss")
ax2.set_ylabel("Log loss", color="tab:red")
plt.title("Choosing k")
plt.savefig("k_selection.png", bbox_inches="tight")
plt.close()

# ---------- 5. Dendrogram at chosen k ----------
_, Z = cluster_genres(genre_matrix, best_k)
plt.figure()
dendrogram(Z)
plt.title("Genre clustering")
plt.savefig("genre_dendrogram.png", bbox_inches="tight")
plt.close()

# ---------- 6. Calibration check for chosen k ----------
labels, _ = cluster_genres(genre_matrix, best_k)
Y = build_combined_genres(genre_matrix, labels, best_k)
model = MultiOutputClassifier(LogisticRegression(max_iter=1000)).fit(X, Y)
probs_j0 = model.estimators_[0].predict_proba(X)[:, 1]
frac_pos, mean_pred = calibration_curve(Y[:, 0], probs_j0, n_bins=10)
plt.figure()
plt.plot(mean_pred, frac_pos, "o-", label="Model")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.legend()
plt.title("Calibration (super-genre 0)")
plt.savefig("calibration.png", bbox_inches="tight")
plt.close()

print("Saved: k_selection.png, genre_dendrogram.png, calibration.png")

Best k by AUC: 11
Saved: k_selection.png, genre_dendrogram.png, calibration.png


In [ ]:
"""
Standalone evaluation of a chosen (k, feature-set) configuration.

Run this AFTER genre_and_feature_optimization.py has told you its
overall_best result. Fill in BEST_K and BEST_FEATURES below with that
output, then run this script on its own.

Answers: "how good is this model, really?" via
  - ROC-AUC and PR-AUC per genre (discrimination)
  - Brier score and log-loss per genre (probability quality)
  - Calibration error (are the probabilities trustworthy?)
  - Lift over a naive base-rate baseline
  - A permutation test for statistical significance
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

CSV_PATH = "data/mxmh_survey_results.csv"   # <-- set this

BEST_K = 13                                  # <-- fill in from the optimization run
BEST_FEATURES = ["Anxiety", "Depression", "Insomnia", "OCD"]  # <-- fill in (mandatory + kept optional)

N_PERMUTATIONS = 50   # how many shuffles for the significance test


# ---------- minimal rebuild of preprocessing / clustering ----------
def preprocess(df):
    feat_df = pd.DataFrame(index=df.index)
    feature_cols = {}
    numeric = ["Anxiety", "Depression", "Insomnia", "OCD", "Age", "Hours per day", "BPM"]
    binary = ["While working", "Instrumentalist", "Composer", "Exploratory", "Foreign languages"]

    for col in numeric:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            if col == "BPM":
                vals = vals.where(vals.between(40, 220))
            feat_df[col] = vals
            feature_cols[col] = [col]

    for col in binary:
        if col in df.columns:
            feat_df[col] = df[col].map({"Yes": 1, "No": 0})
            feature_cols[col] = [col]

    if "Primary streaming service" in df.columns:
        service = df["Primary streaming service"].fillna("Missing").astype(str)
        dummies = pd.get_dummies(service, prefix="service")
        feat_df = pd.concat([feat_df, dummies], axis=1)
        feature_cols["Primary streaming service"] = list(dummies.columns)

    feat_df = feat_df.fillna(feat_df.median(numeric_only=True))
    return feat_df, feature_cols


def build_matrix(feat_df, feature_cols, chosen):
    cols = [c for feat in chosen for c in feature_cols[feat]]
    return feat_df[cols].values


def cluster_genres(genre_matrix, k):
    corr = np.nan_to_num(np.corrcoef(genre_matrix.T))
    dist = np.clip(1 - corr, 0, None)
    np.fill_diagonal(dist, 0)
    Z = linkage(squareform(dist, checks=False), method="average")
    return fcluster(Z, t=k, criterion="maxclust")


def build_combined_genres(genre_matrix, labels):
    uniq = np.unique(labels)
    combined = np.zeros((genre_matrix.shape[0], len(uniq)))
    for idx, c in enumerate(uniq):
        cols = np.where(labels == c)[0]
        combined[:, idx] = (genre_matrix[:, cols].sum(axis=1) > 0).astype(int)
    return combined


# ---------- out-of-fold predictions (model + baseline) ----------
def oof_predictions(X, Y, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    model_probs = np.zeros_like(Y, dtype=float)
    baseline_probs = np.zeros_like(Y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        scaler = StandardScaler().fit(X[train_idx])
        Xtr, Xte = scaler.transform(X[train_idx]), scaler.transform(X[test_idx])
        model = MultiOutputClassifier(LogisticRegression(max_iter=1000))
        model.fit(Xtr, Y[train_idx])
        for j, est in enumerate(model.estimators_):
            model_probs[test_idx, j] = est.predict_proba(Xte)[:, 1]
            baseline_probs[test_idx, j] = Y[train_idx, j].mean()  # naive base-rate predictor

    return model_probs, baseline_probs


def per_genre_metrics(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return None
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "prevalence": y_true.mean(),
    }


def permutation_test(X, Y, n_perm=N_PERMUTATIONS, seed=0):
    rng = np.random.default_rng(seed)
    real_probs, _ = oof_predictions(X, Y)
    real_aucs = [roc_auc_score(Y[:, j], real_probs[:, j])
                 for j in range(Y.shape[1]) if len(np.unique(Y[:, j])) > 1]
    real_mean = np.mean(real_aucs)

    null_means = []
    for _ in range(n_perm):
        Y_shuffled = Y[rng.permutation(Y.shape[0])]
        probs, _ = oof_predictions(X, Y_shuffled)
        aucs = [roc_auc_score(Y_shuffled[:, j], probs[:, j])
                for j in range(Y_shuffled.shape[1]) if len(np.unique(Y_shuffled[:, j])) > 1]
        if aucs:
            null_means.append(np.mean(aucs))

    p_value = np.mean([n >= real_mean for n in null_means])
    return real_mean, null_means, p_value


def interpret_auc(auc):
    if auc < 0.55:
        return "no better than chance"
    if auc < 0.65:
        return "weak signal"
    if auc < 0.75:
        return "decent, usable signal"
    return "strong signal"


def main():
    df = pd.read_csv(CSV_PATH)
    freq_cols = [c for c in df.columns if c.startswith("Frequency [")]
    genre_matrix = (df[freq_cols] != "Never").astype(int).values

    feat_df, feature_cols = preprocess(df)
    labels = cluster_genres(genre_matrix, BEST_K)
    Y = build_combined_genres(genre_matrix, labels)
    X = build_matrix(feat_df, feature_cols, BEST_FEATURES)

    model_probs, baseline_probs = oof_predictions(X, Y)

    print(f"{'Genre':<8}{'n_pos':>7}{'AUC':>8}{'Base AUC':>10}{'PR-AUC':>9}{'Brier':>9}{'LogLoss':>10}  Verdict")
    rows = []
    for j in range(Y.shape[1]):
        m = per_genre_metrics(Y[:, j], model_probs[:, j])
        b = per_genre_metrics(Y[:, j], baseline_probs[:, j])
        if m is None:
            continue
        rows.append((j, m, b))
        print(f"{j:<8}{int(Y[:,j].sum()):>7}{m['roc_auc']:>8.3f}{b['roc_auc']:>10.3f}"
              f"{m['pr_auc']:>9.3f}{m['brier']:>9.3f}{m['log_loss']:>10.3f}  {interpret_auc(m['roc_auc'])}")

    macro_auc = np.mean([m["roc_auc"] for _, m, _ in rows])
    macro_base_auc = np.mean([b["roc_auc"] for _, _, b in rows])
    macro_brier = np.mean([m["brier"] for _, m, _ in rows])
    print(f"\nMacro-average AUC: {macro_auc:.3f}  (baseline: {macro_base_auc:.3f})")
    print(f"Macro-average Brier score: {macro_brier:.3f}  (lower is better; 0.25 = uninformative coin-flip)")

    real_mean, null_means, p_value = permutation_test(X, Y)
    print(f"\nPermutation test: real mean AUC = {real_mean:.3f}, "
          f"null mean AUC = {np.mean(null_means):.3f} +/- {np.std(null_means):.3f}, "
          f"p-value = {p_value:.3f}")
    if p_value < 0.05:
        print("-> The model's discrimination is unlikely to be due to chance.")
    else:
        print("-> Cannot rule out that this performance is due to chance at k=%d with these features." % BEST_K)

    # ---------- bar chart: per-genre AUC vs baseline ----------
    genre_ids = [r[0] for r in rows]
    aucs = [r[1]["roc_auc"] for r in rows]
    base_aucs = [r[2]["roc_auc"] for r in rows]
    x = np.arange(len(genre_ids))
    plt.figure(figsize=(9, 5))
    plt.bar(x - 0.2, aucs, width=0.4, label="Model")
    plt.bar(x + 0.2, base_aucs, width=0.4, label="Base-rate baseline")
    plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    plt.xticks(x, [f"G{g}" for g in genre_ids])
    plt.ylabel("ROC-AUC")
    plt.title(f"Per-genre AUC: model vs. baseline (k={BEST_K})")
    plt.legend()
    plt.savefig("model_vs_baseline_auc.png", bbox_inches="tight")
    plt.close()
    print("\nSaved: model_vs_baseline_auc.png")


if __name__ == "__main__":
    main()


Genre     n_pos     AUC  Base AUC   PR-AUC    Brier   LogLoss  Verdict
0           429   0.578     0.476    0.638    0.239     0.671  weak signal
1           456   0.507     0.462    0.619    0.238     0.669  no better than chance
2           500   0.528     0.452    0.683    0.219     0.629  no better than chance
3           320   0.448     0.427    0.399    0.250     0.693  no better than chance
4           293   0.517     0.464    0.405    0.242     0.677  no better than chance
5           646   0.568     0.447    0.894    0.107     0.373  weak signal
6           680   0.532     0.419    0.930    0.071     0.272  no better than chance
7           655   0.631     0.456    0.923    0.095     0.335  weak signal
8           567   0.485     0.453    0.763    0.178     0.543  no better than chance
9           475   0.439     0.466    0.596    0.232     0.658  no better than chance
10          393   0.499     0.449    0.531    0.251     0.696  no better than chance
11          444   0.535 

In [40]:
"""
Trains the final model (fixed k and feature set, no more CV/search)
on the FULL dataset and exports everything needed to reuse it later:
  - the fitted MultiOutputClassifier
  - the fitted StandardScaler
  - which columns each feature maps to (for one-hot features)
  - which original genres belong to each combined genre (for interpretability)

Fill in BEST_K and BEST_FEATURES from your optimization run, then run
this once. It writes music_genre_model.joblib.
"""

import numpy as np
import pandas as pd
import joblib
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

CSV_PATH = "data/mxmh_survey_results.csv"     # <-- set this
OUTPUT_PATH = "music_genre_model.joblib"

BEST_K = 13                                    # <-- fill in from the optimization run
BEST_FEATURES = ["Anxiety", "Depression", "Insomnia", "OCD"]  # <-- fill in


def preprocess(df):
    feat_df = pd.DataFrame(index=df.index)
    feature_cols = {}
    numeric = ["Anxiety", "Depression", "Insomnia", "OCD", "Age", "Hours per day", "BPM"]
    binary = ["While working", "Instrumentalist", "Composer", "Exploratory", "Foreign languages"]

    for col in numeric:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            if col == "BPM":
                vals = vals.where(vals.between(40, 220))
            feat_df[col] = vals
            feature_cols[col] = [col]

    for col in binary:
        if col in df.columns:
            feat_df[col] = df[col].map({"Yes": 1, "No": 0})
            feature_cols[col] = [col]

    if "Primary streaming service" in df.columns:
        service = df["Primary streaming service"].fillna("Missing").astype(str)
        dummies = pd.get_dummies(service, prefix="service")
        feat_df = pd.concat([feat_df, dummies], axis=1)
        feature_cols["Primary streaming service"] = list(dummies.columns)

    feat_df = feat_df.fillna(feat_df.median(numeric_only=True))
    return feat_df, feature_cols


def build_matrix(feat_df, feature_cols, chosen):
    cols = [c for feat in chosen for c in feature_cols[feat]]
    return feat_df[cols].values, cols


def cluster_genres(genre_matrix, k):
    corr = np.nan_to_num(np.corrcoef(genre_matrix.T))
    dist = np.clip(1 - corr, 0, None)
    np.fill_diagonal(dist, 0)
    Z = linkage(squareform(dist, checks=False), method="average")
    return fcluster(Z, t=k, criterion="maxclust")


def build_combined_genres(genre_matrix, labels):
    uniq = np.unique(labels)
    combined = np.zeros((genre_matrix.shape[0], len(uniq)))
    for idx, c in enumerate(uniq):
        cols = np.where(labels == c)[0]
        combined[:, idx] = (genre_matrix[:, cols].sum(axis=1) > 0).astype(int)
    return combined, uniq


def main():
    df = pd.read_csv(CSV_PATH)
    freq_cols = [c for c in df.columns if c.startswith("Frequency [")]
    genre_names = [c.replace("Frequency [", "").rstrip("]") for c in freq_cols]
    genre_matrix = (df[freq_cols] != "Never").astype(int).values

    feat_df, feature_cols = preprocess(df)
    labels = cluster_genres(genre_matrix, BEST_K)
    Y, cluster_ids = build_combined_genres(genre_matrix, labels)
    X, input_columns = build_matrix(feat_df, feature_cols, BEST_FEATURES)

    # map each combined genre back to its original genre names, for readability
    cluster_map = {
        f"combined_genre_{idx}": [genre_names[i] for i in np.where(labels == c)[0]]
        for idx, c in enumerate(cluster_ids)
    }

    scaler = StandardScaler().fit(X)
    X_scaled = scaler.transform(X)
    model = MultiOutputClassifier(LogisticRegression(max_iter=1000))
    model.fit(X_scaled, Y)

    bundle = {
        "model": model,
        "scaler": scaler,
        "input_features": BEST_FEATURES,   # order matters at predict time
        "input_columns": input_columns,    # expanded column order after one-hot
        "feature_cols": feature_cols,      # needed to re-expand one-hot features later
        "k": BEST_K,
        "cluster_map": cluster_map,        # combined_genre_i -> [original genres]
    }
    joblib.dump(bundle, OUTPUT_PATH)
    print(f"Saved model bundle to {OUTPUT_PATH}")
    print("\nCombined genres:")
    for name, members in cluster_map.items():
        print(f"  {name}: {members}")


if __name__ == "__main__":
    main()

Saved model bundle to music_genre_model.joblib

Combined genres:
  combined_genre_0: ['EDM']
  combined_genre_1: ['Lofi']
  combined_genre_2: ['Video game music']
  combined_genre_3: ['K pop']
  combined_genre_4: ['Latin']
  combined_genre_5: ['Hip hop', 'R&B', 'Rap']
  combined_genre_6: ['Pop']
  combined_genre_7: ['Metal', 'Rock']
  combined_genre_8: ['Classical']
  combined_genre_9: ['Jazz']
  combined_genre_10: ['Country']
  combined_genre_11: ['Folk']
  combined_genre_12: ['Gospel']
